In [ ]:
%matplotlib inline
import flopy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pathlib as pl
import pandas as pd
import sys
import itertools
import geopandas as gpd
import xarray as xr
from osgeo import gdal
import rasterio
import xugrid
from pathlib import Path

In [ ]:
sys.path.append("../common")
from liss_settings import cx, cx_provider, extent, boxx, boxy, extentmax, fig_ext, transparent

# Units

In [ ]:
# units = "mm"
# conversion_factor = 1.0
# if units == "mm":
#     conversion_factor = 25.4
# total_key = f"total_{units}"

# Import Models

## Uncoupled base model

In [ ]:
base_ws = "../modflow/pj_2018_adjust_FINAL/base/"
sim = flopy.mf6.MFSimulation.load(sim_ws=base_ws, verbosity_level=0)
gwf = sim.get_model("gwf")
# # Number of GHB cells 
nghb = gwf.ghb.stress_period_data.get_dataframe()[0].shape[0]
nchd = gwf.get_package("chd_coast").stress_period_data.get_dataframe()[0].shape[0]

In [ ]:
mg =gwf.modelgrid
mg_gdf = mg.geo_dataframe 
# Area of one cell
area = mg.delr[0]*mg.delc[0]
print(f'Cell area: {mg.delr[0]*mg.delc[0]} {mg.units}^2')
mg_gdf['row_pj'] = [x[0] for x in itertools.product(range(mg.nrow), range(mg.ncol))]
mg_gdf['col_pj']=  [x[1] for x in itertools.product(range(mg.nrow), range(mg.ncol))]
mg_gdf.set_crs(epsg = 4456, inplace=True)
print(mg_gdf.crs)

In [ ]:
path = pl.Path(base_ws)/'external'
lay_arr_child = np.array([np.loadtxt(path / 'top.dat')] + [np.loadtxt(file) for file in sorted(path.glob('bot*.dat'))])

In [ ]:
tdis = sim.tdis
mf_startdate = tdis.start_date_time.array
mf_perioddata= tdis.perioddata.array
# calculate date for each stress period
mf_SPdates = [pd.to_datetime(mf_startdate)]
for perlen, nstp, tsmult in mf_perioddata:
    mf_SPdates.append(mf_SPdates[-1] + pd.Timedelta(days = perlen))
# remove the last date because it is extra
mf_SPdates = mf_SPdates[:-1]
# Compute cumulative total times at the end of each stress period
mf_totaltimes= tdis.perioddata.array['perlen'].cumsum()

mf_tdis_df = pd.DataFrame({'Date':mf_SPdates,
                           'SP': list(mf_perioddata),
                           'totim': mf_totaltimes})

In [ ]:
lst_ucoupled= gwf.output.list().get_dataframes()[0]
# Reset the index
lst_ucoupled.reset_index(inplace=True)
lst_ucoupled['date'] = mf_SPdates[:len(lst_ucoupled)]

## Uncoupled, NOAA time varying CHD

In [ ]:
uncoupled_noaa_ws = r'C:\Users\bbayrakt\OneDrive - DOI\LISS_GW\GW_Models\LISUS_conditionedmodels_BNB\2018\conditioned_model_2018_daily\_mfsetup\models\pj_2018_adjust_FINAL_timevaryCHD'
sim = flopy.mf6.MFSimulation.load(sim_ws=uncoupled_noaa_ws ,load_only = ['dis'], verbosity_level=0)
gwf_uncoupled_noaa = sim.get_model("gwf")

lst_uncoupled_noaa= gwf_uncoupled_noaa.output.list().get_dataframes()[0]
lst_uncoupled_noaa.reset_index(inplace=True)
lst_uncoupled_noaa['date'] = mf_SPdates[:len(lst_uncoupled_noaa)]

## Coupled base model

In [ ]:
coupled_ws = '../modflow/pj_2018_adjust_FINAL/MF+DFlowNEWGRID/run_15.00M'
sim = flopy.mf6.MFSimulation.load(sim_ws=coupled_ws, verbosity_level=0)
gwf_coupled = sim.get_model("gwf")

lst_coupled= gwf_coupled.output.list().get_dataframes()[0]
lst_coupled.reset_index(inplace=True)
lst_coupled['date'] = mf_SPdates[:len(lst_coupled)]

## Coupled, No recharge

In [ ]:
ws_norech = '../modflow/pj_2018_adjust_FINAL_NoRecharge/MF+DFlowNEWGRID/run_15.00M'
sim = flopy.mf6.MFSimulation.load(sim_ws=ws_norech, load_only = ['dis'],verbosity_level=0)
gwf_coupled_norech = sim.get_model("gwf")

lst_coupled_norech= gwf_coupled_norech.output.list().get_dataframes()[0]
lst_coupled_norech.reset_index(inplace=True)
lst_coupled_norech['date'] = mf_SPdates[:len(lst_coupled_norech)]

## Coupled, No Storm Surge

In [ ]:
ws_nosurge = '../modflow/pj_2018_adjust_FINAL/MF+DFlowNEWGRID_NoStormSurge/run_15.00M'
sim = flopy.mf6.MFSimulation.load(sim_ws=ws_nosurge, load_only = ['dis'],verbosity_level=0)
gwf_coupled_nosurge = sim.get_model("gwf")

lst_coupled_nosurge= gwf_coupled_nosurge.output.list().get_dataframes()[0]
lst_coupled_nosurge.reset_index(inplace=True)
lst_coupled_nosurge['date'] = mf_SPdates[:len(lst_coupled_nosurge)]

# Coastal Exchange

Figure Settings

In [ ]:
ws = pl.Path("figures")
ws.mkdir(exist_ok=True, parents=True)

In [ ]:
# # Create labels
# # ===================
# labels = [value.split(sep="_")[1] for value in sim_dirs]

# for idx, label in enumerate(labels):
#     s = f"{float(label[:5]):>2.0f}" + " " + label[-1]
#     if s.endswith(" D"):
#         s = s.replace(" D", " Day ")
#     elif s.endswith(" H"):
#         s = s.replace(" H", " Hour")
#     elif s.endswith(" M"):
#         s = s.replace(" M", " Min.")
#     labels[idx] = s    
# labels

In [ ]:
line_styles = ["-", "--", "-.", ":", (0, (3, 10, 1, 10, 1, 10)), (0, (3, 1, 1, 1, 1, 1))]

In [ ]:
colors = [value for key, value in mcolors.TABLEAU_COLORS.items()]

## GHB

### Observation Cells

In [ ]:
# GHB Obs Cells
# #=====================================================
obscells_GHB = pd.read_csv(fr"{base_ws}gwf.ghb.obs",skiprows=5, sep=r'  ', names = ['ghb_name','head','cellid'])
obscells_GHB['cellid'] = obscells_GHB['cellid'].str.replace(r'\s+', ',', regex=True)
# Split into new columns
obscells_GHB[['lay', 'row', 'col']] = obscells_GHB['cellid'].str.split(',', expand=True)
obscells_GHB = obscells_GHB.iloc[:-1]
obscells_GHB[['lay', 'row', 'col']] = obscells_GHB[['lay', 'row', 'col']].astype(int)
obscells_GHB[['lay', 'row', 'col']]-=1

### Observation Output

In [ ]:

obs_path = Path(coupled_ws) / "outputs" /"GHB_obs.csv"
print(obs_path)
# Load the GHB observations
obs_ghb = flopy.utils.Mf6Obs(obs_path).get_dataframe(start_datetime="1-1-2017")
# Calculate the length of each timestep
obs_ghb["delt (days)"] = obs_ghb["totim"].diff()
# Replace the first rows time step length because it will be nan
obs_ghb.loc[obs_ghb["delt (days)"].isnull(), "delt (days)"] = obs_ghb["delt (days)"].iloc[1]

# Calculate Volumes of exchange per timestep, and cumualtive volumes
#--------------------------------------------
# list of GHB observation names
obs_cols = [c for c in obs_ghb.columns if "GHB" in c]
obs_ghb['ft^3'] = obs_ghb[obs_cols].sum(axis=1) * obs_ghb["delt (days)"]
obs_ghb['ft^3/day'] = obs_ghb[obs_cols].sum(axis=1)
obs_ghb['CUM_ft^3'] = obs_ghb["ft^3"].cumsum()
obs_ghb['ft'] = obs_ghb['ft^3'] /(nghb*area)
obs_ghb['ft/day'] = obs_ghb['ft^3/day'] /(nghb*area)
obs_ghb["CUM_ft"] = obs_ghb["ft"].cumsum()
obs_ghb["ZERO"] = 0.

print(fr'GHB observation file...')
display(obs_ghb.head())

In [ ]:
#     #df.drop("delt (days)", axis=1, inplace=True)
#     # number of timesteps
#     obsdict_GHB[key]["ntimes"] = df.shape[0]
#     # store DF
#     obsdict_GHB[key]["df"] = df.copy()
#     # Final cumulative total 
#     obsdict_GHB[key]["total volume"] = df["CUM_ft^3"].iloc[-1]

# print(fr'GHB observation file for {key}...')
# display(df.head())

## CHD

### Observation Cells

In [ ]:
# CHD Obs Cells
# #=====================================================
obscells_CHD = pd.read_csv(fr"{base_ws}gwf_0.chd.obs",skiprows=5, sep=r'  ', names = ['chd_name','head','cellid'])
obscells_CHD['cellid'] = obscells_CHD['cellid'].str.replace(r'\s+', ',', regex=True)
obscells_CHD[['lay', 'row', 'col']] = obscells_CHD['cellid'].str.split(',', expand=True)
obscells_CHD = obscells_CHD.iloc[:-1]
obscells_CHD[['lay', 'row', 'col']] = obscells_CHD[['lay', 'row', 'col']].astype(int)
obscells_CHD[['lay', 'row', 'col']]-=1

#### Split up obs cells based on how you want to analyze them

In [ ]:
# Split up obs cells based on how you want to analyze them
#-===================================================
# all CHDs on the surface
obscells_CHDsurface = obscells_CHD[obscells_CHD['lay'] == 0]
# all perimeter CHDs on the coast
obscells_CHDperimeter  = obscells_CHD[obscells_CHD['chd_name'].str.contains('perimeter')]
# perimeter CHDs on the coast that are not in layer 1
obscells_CHDdepth= obscells_CHD[obscells_CHD['lay'] != 0]

# Determine the observation names in order to drop column names later
obsname_CHDdepth = set(obscells_CHD.loc[obscells_CHDdepth.index, 'chd_name'].str.upper().str.replace("'", ""))
obsname_CHDsurface = set(obscells_CHD.loc[obscells_CHDsurface.index, 'chd_name'].str.upper().str.replace("'", ""))
obsname_perimeter = set(obscells_CHD.loc[obscells_CHDperimeter.index, 'chd_name'].str.upper().str.replace("'", ""))

### Observation Output

#### All

In [ ]:
# Save ALL CHD observation results for all simulations and al
# ======================================
obs_path = Path(coupled_ws) / "outputs" /"chd_coastal_obs.csv"
print(f'loading:{obs_path}')
obs_CHD = flopy.utils.Mf6Obs(obs_path).get_dataframe(start_datetime="1-1-2017")
all_obs_cols = [c for c in obs_CHD.columns if c != "totim"]

# Calculate timestep length safely
obs_CHD["delt (days)"] = obs_CHD["totim"].diff().fillna(method="bfill")


In [ ]:
print(len(obs_CHD.columns))

#### Divided up

In [ ]:
# Filter Out CHD Observations into groups 
#=============================================
# ---------- Surface Coastal Observations ----------
# drop the cells in layers >0
obs_CHD_surface = obs_CHD.drop(columns=obsname_CHDdepth, errors='ignore')
# filter the all CHD observation columns, to the ones of interest 
obs_CHD_surface['ft^3'] = obs_CHD_surface[list(obsname_CHDsurface)].sum(axis=1)* obs_CHD_surface["delt (days)"]
obs_CHD_surface['ft^3/day'] = obs_CHD_surface[list(obsname_CHDsurface)].sum(axis=1)
obs_CHD_surface['CUM_ft^3'] = obs_CHD_surface["ft^3"].cumsum()
obs_CHD_surface['ft'] = obs_CHD_surface['ft^3'] /(nghb*area)
obs_CHD_surface['ft/day'] = obs_CHD_surface['ft^3/day'] /(nghb*area)
obs_CHD_surface["CUM_ft"] = obs_CHD_surface["ft"].cumsum()
print(len(obs_CHD_surface.columns))



In [ ]:

# ---------- Surface Coastal Observations - No Perimeter ----------
# Drops all perimeter cells 
drop_cols = obsname_CHDdepth.union(obsname_perimeter)
obs_CHD_surface_noperim = obs_CHD.drop(columns=drop_cols, errors='ignore')
obs_cols = [c for c in obs_CHD_surface_noperim.columns if "BAY_" in c]
obs_CHD_surface_noperim['ft^3'] = obs_CHD_surface_noperim[obs_cols].sum(axis=1)* obs_CHD_surface_noperim["delt (days)"]
obs_CHD_surface_noperim['ft^3/day'] = obs_CHD_surface_noperim[obs_cols].sum(axis=1)
obs_CHD_surface_noperim['CUM_ft^3'] = obs_CHD_surface_noperim["ft^3"].cumsum()
obs_CHD_surface_noperim['ft'] = obs_CHD_surface_noperim['ft^3'] /(nghb*area)
obs_CHD_surface_noperim['ft/day'] = obs_CHD_surface_noperim['ft^3/day'] /(nghb*area)
obs_CHD_surface_noperim["CUM_ft"] = obs_CHD_surface_noperim["ft"].cumsum()
print(len(obs_CHD_surface_noperim.columns))


In [ ]:

# ---------- Depth Coastal Observations ----------
# drop all surface cells
obs_CHD_atDepth = obs_CHD.drop(columns=obsname_CHDsurface, errors='ignore')
obs_CHD_atDepth['ft^3'] = obs_CHD_atDepth[list(obsname_CHDdepth)].sum(axis=1)* obs_CHD_atDepth["delt (days)"]
obs_CHD_atDepth['ft^3/day'] = obs_CHD_atDepth[list(obsname_CHDdepth)].sum(axis=1)
obs_CHD_atDepth['CUM_ft^3'] = obs_CHD_atDepth["ft^3"].cumsum()
obs_CHD_atDepth['ft'] = obs_CHD_atDepth['ft^3'] /(nghb*area)
obs_CHD_atDepth['ft/day'] = obs_CHD_atDepth['ft^3/day'] /(nghb*area)
obs_CHD_atDepth["CUM_ft"] = obs_CHD_atDepth["ft"].cumsum()
print(len(obs_CHD_atDepth.columns))


## Figure

### Coastal Exchange time series

In [ ]:
#=================================================================
start_date = pd.Timestamp('2018-09-19 00:00:00')

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(
        layout="constrained",
        figsize=(8,8),
        )

# ax.set_ylim(-1 * conversion_factor, 1 * conversion_factor)
df_ghb = obs_ghb
df_ghb["ft^3/day"].plot(ax=ax, lw=0.75, ls="-", color='red',sharex=True, label= "GHB")
df_ghb["ZERO"].plot(ax=ax, lw=0.5, ls=":", color="black", sharex=True)

df_chd = obs_CHD_surface_noperim
df_chd['ft^3/day'].plot(ax=ax, lw=0.75, ls="-", color='blue', sharex=True, label ="CHD - no perimeter cells")
#df_chd["ZERO"].plot(ax=ax, lw=0.5, ls="--", color="black", sharex=True)

df_chd = obs_CHD_surface
df_chd['ft^3/day'].plot(ax=ax, lw=0.75, ls="--", color='blue', sharex=True, label ="CHD- with perimeter cells")
#df_chd["ZERO"].plot(ax=ax, lw=0.5, ls="--", color="black", sharex=True)

ax.set_ylabel(f"ft^3/day")
ax.set_xlabel("")
ax.set_xlim(start_date, df_ghb.index[-1])
flopy.plot.styles.heading(ax, x=0.2)
fig.suptitle('Costal Exchange (ft^3/day)')
leg = flopy.plot.styles.graph_legend(ax=ax, loc="upper left", title="none", ncol=2)


In [ ]:
#=================================================================
start_date = pd.Timestamp('2018-09-19 00:00:00')

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(8,8),
        layout="constrained"
    )

# =========================
# TOP: Normalized (ft/day)
# =========================
df_ghb = obs_ghb
df_ghb["ft/day"].plot(ax=ax[0], lw=0.75, ls="-", color='red', label="GHB")

df_chd = obs_CHD_surface_noperim
df_chd["ft/day"].plot(ax=ax[0], lw=0.75, ls="-", color='blue', label="CHD - no perimeter")

df_chd = obs_CHD_surface
df_chd["ft/day"].plot(ax=ax[0], lw=0.75, ls="--", color='blue', label="CHD - with perimeter")

df_ghb["ZERO"].plot(ax=ax[0], lw=0.5, ls=":", color="black")

ax[0].set_title('Exchange normalized by cell area')
ax[0].set_ylabel("ft/day")
ax[0].set_xlim(start_date, df_ghb.index[-1])
ax[0].set_xlabel("")


# =========================
# BOTTOM: Raw (ft³/day)
# =========================
df_ghb = obs_ghb
df_ghb["ft^3/day"].plot(ax=ax[1], lw=0.75, ls="-", color='red', label="GHB")

df_chd = obs_CHD_surface_noperim
df_chd["ft^3/day"].plot(ax=ax[1], lw=0.75, ls="-", color='blue', label="CHD - no perimeter")

df_chd = obs_CHD_surface
df_chd["ft^3/day"].plot(ax=ax[1], lw=0.75, ls="--", color='blue', label="CHD - with perimeter")

df_ghb["ZERO"].plot(ax=ax[1], lw=0.5, ls=":", color="black")

ax[1].set_title('Exchange raw values')
ax[1].set_ylabel("ft^3/day")
ax[1].set_xlim(start_date, df_ghb.index[-1])
ax[1].set_xlabel("")


# =========================
# Titles + legend
# =========================
fig.suptitle('Coastal Exchange\nWith and Without Perimeter Cells')

flopy.plot.styles.graph_legend(
    ax=ax[1],
    loc="upper left",
    title="none",
    ncol=2
)

### Coastal Exchange Map

#### With Perimeter Cells

In [ ]:
sim = 'run_15.00M'
datetimes  = ("2018-09-29 00:00:00.000000",
    "2018-09-29 06:00:00.000000"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i, datetime in enumerate(datetimes):
    # GHB 
    # ==========
    #Populate map with GHB values
    mg_coastal = mg_gdf.copy()
    mg_coastal['coastal_flux'] = float(0)
 
    #Populate map with GHB values
    for r,c in zip(obscells_GHB['row'], obscells_GHB['col']):
        # identify the ghb cell name from the row,col
        name = obscells_GHB.loc[((obscells_GHB['row'] == r) & (obscells_GHB['col'] == c)), 'ghb_name'].iloc[0]
        # use the ghb name to correctly index the observation results
        mg_coastal.loc[((mg_coastal['row_pj'] == r) & (mg_coastal['col_pj'] == c)),'coastal_flux'] = obs_ghb[name.upper()].loc[datetime]

    for name,r,c in zip(obscells_CHDsurface['chd_name'],obscells_CHDsurface['row'], obscells_CHDsurface['col']):
        name = name.upper().replace("'","")
        mg_coastal.loc[((mg_coastal['row_pj'] == r) & (mg_coastal['col_pj'] == c)),'coastal_flux'] = obs_CHD_surface[name].loc[datetime]

    # Populate Figure
    #---------
    ax = axes[i]
    vmin = mg_coastal[mg_coastal['coastal_flux']!=0]['coastal_flux'].min()
    vmax = mg_coastal[mg_coastal['coastal_flux']!=0]['coastal_flux'].max()
    absmax = max(abs(vmin),abs(vmax))
    mg_coastal[mg_coastal['coastal_flux']!=0].plot(
        column = 'coastal_flux',
        cmap='seismic_r',
        vmin=-absmax,
        vmax=absmax,
        ax=ax,
        legend=True,
        legend_kwds={'shrink': 0.6}
    )
    #cx.add_basemap(ax=ax, crs=mg_coastal.crs)
    if datetime =="2018-09-29 00:00:00.000000":
        tide = "Low Tide"
        ax.set_title(f'{datetime}\n{tide}')

    if datetime =="2018-09-29 06:00:00.000000":
        tide = "High Tide"
        ax.set_title(f'{datetime}\n{tide}')

fig.suptitle(f'Coastal Flux (ft^3/day) \n coupling: {sim}')
fig.tight_layout()
plt.show()


# Head Observations

In [ ]:
lines_uncoupled = ':'
lines_coupled = '-'
lines_coupled_norech = '-.'
lines_uncoupled_timevary = '--'
site_colors = ['red', 'darkorange', 'gold','yellowgreen', 'deepskyblue','darkviolet', 'hotpink']


## load

In [ ]:
# GWF Obs Cells
# #=====================================================
obscells_heads = pd.read_csv(fr"{base_ws}\gwf.obs",skiprows=5, sep=r'  ', names = ['obs_name','head','cellid'])
# Replace *any* whitespace in cellid with commas
obscells_heads['cellid'] = obscells_heads['cellid'].str.replace(r'\s+', ',', regex=True)
# Split into new columns
obscells_heads[['lay', 'row', 'col']] = obscells_heads['cellid'].str.split(',', expand=True)
obscells_heads = obscells_heads.iloc[:-1]
obscells_heads[['lay', 'row', 'col']] = obscells_heads[['lay', 'row', 'col']].astype(int)
obscells_heads[['lay', 'row', 'col']]-=1


In [ ]:
# Uncoupled, NOAA CHD
path = fr'{uncoupled_noaa_ws}/outputs/hd_obs.csv'
print(path)
headobs_uncoupled_NOAACHD = flopy.utils.Mf6Obs(path).get_dataframe(start_datetime="1-1-2017")

In [ ]:
# Uncoupled, Base
path = fr'{base_ws}/outputs/hd_obs.csv'
print(path)
headobs_uncoupled = flopy.utils.Mf6Obs(path).get_dataframe(start_datetime="1-1-2017")

# Uncoupled, NOAA CHD
path = fr'{uncoupled_noaa_ws}/outputs/hd_obs.csv'
print(path)
headobs_uncoupled_NOAACHD = flopy.utils.Mf6Obs(path).get_dataframe(start_datetime="1-1-2017")

# Coupled model
path = fr'{coupled_ws}/outputs/hd_obs.csv'
print(path)
headobs_coupled = flopy.utils.Mf6Obs(path).get_dataframe(start_datetime="1-1-2017")

# Coupled model, no storm surge 
path = fr'{ws_nosurge}/outputs/hd_obs.csv'
print(path)
headobs_coupled_nosurge = flopy.utils.Mf6Obs(path).get_dataframe(start_datetime="1-1-2017")

# Coupled model, no rech
path = fr'{ws_norech}/outputs/hd_obs.csv'
print(path)
headobs_coupled_norech = flopy.utils.Mf6Obs(path).get_dataframe(start_datetime="1-1-2017")

## Observed vs. Simulated in PJ

In [ ]:
pj_network = gpd.read_file(r"C:\Users\bbayrakt\OneDrive - DOI\LISS_GW\GIS\PJ_mininetwork_2024\PJ_mininetwork_withsensors.shp")
pj_network.to_crs(epsg=4456,inplace=True)
assert pj_network.crs == mg_gdf.crs

# 'screen_top' is the depth to the screen from the casing 
# 'casing_top' is in reference to ft above msl
# calculate 'screen_top_msl' which is in reference to sea-level, by su
pj_network['screen_top_msl'] = pj_network['casing_top']- pj_network['screen_top']

# Determine where the wells intersect the model grid
#---------------------------------------------
pj_network_modelcells = gpd.sjoin(mg_gdf,pj_network,how = 'inner', predicate='intersects').sort_values('Site')
site_obsnames = {}
for idx, (sitename, screentop, r, c) in enumerate(zip(
pj_network_modelcells['Site'],pj_network_modelcells['screen_top_msl'],pj_network_modelcells['row_pj'],pj_network_modelcells['col_pj'])):
    if not np.isnan(screentop):
        stack = lay_arr_child[:, r, c]
        l = (np.abs(stack - screentop)).argmin()
        print(f"{sitename}: screen top elevation ={screentop}, layer={l}, layer_val={stack[l]}")
    else:
        if sitename == 'Marina':
            l, r, c = 0, 66, 60
           # l, r, c = 0, r,c
        if sitename == 'MillCreek':
            l, r, c = 0, r, c

    obs_name = obscells_heads.loc[(obscells_heads['lay'] == l) &(obscells_heads['row'] == r) &(obscells_heads['col'] == c), 'obs_name'].iloc[0].replace("'", "")
    site_obsnames[sitename] ={'obs_name':obs_name, 
                              'cellid':(l,r,c)}

# IMPORT AND SAVE PJ WATER LEVEL OBSERVATIONS
# =================================================
from pathlib import Path
lai_dict = {}
# GROUNDWATER
path = Path('../../../PJ_2024WaterLevelNetwork/data/gw')
for file in path.glob('*.csv'):
    f =pd.read_csv(file)
    f['Date/Time'] = pd.to_datetime(f['Date/Time'])
    lai_dict[file.name[:-4]] = f
# SURFACE WATER
path = Path('../../../PJ_2024WaterLevelNetwork/data/sw')
for file in path.glob('*.csv'):
    f =pd.read_csv(file)
    lai_dict[file.name[:-4]] = f
    f['Time'] = pd.to_datetime(f['Time'])


In [ ]:
x_colname = 'Time'
y_colname = 'elevation'
series = headobs_coupled[site_obsnames['Marina']['obs_name'].upper()]

fig, ax = plt.subplots()
ax.plot(series.values, color='k', label = "coupled model")

lai_dict['Marina'][y_colname].plot(ax=ax, color = 'red', label = 'observed')
ax.set_xlim(0,5000)
ax.legend()

In [ ]:
sim = 'run_15.00M'
colors = ['red', 'darkorange', 'gold','yellowgreen', 'deepskyblue','darkviolet', 'hotpink']
start_date = pd.Timestamp('2018-09-18 00:00:00')
end_date = pd.Timestamp('2018-10-18 00:00:00')
fig, ax = plt.subplots(1, 3, figsize=(12, 6), constrained_layout=True)
# Plot site map
#==================
# pj_network_modelcells.plot(ax=ax[0])
# pj_network.plot(column='Site',  ax=ax[0])
# cx.add_basemap(ax=ax[0], crs=pj_network.crs)


# SIMULATED HEADS
#-------------------------
for idx, (sitename,value) in enumerate(site_obsnames.items()):
    l,r,c = value['cellid']
    obs_name = value['obs_name']
    print(sitename, obs_name, l, r, c)
    print('-----------------')
    if sitename in['MillCreek','Marina']:
        x_colname = 'Time'
        y_colname = 'elevation'
    else:
        x_colname = 'Date/Time'
        y_colname = 'WLE estimate'


    mg_gdf[(mg_gdf['row_pj'] == r) & (mg_gdf['col_pj'] == c)].plot(ax=ax[0],color = colors[idx])
    pj_network.loc[pj_network['Site'] == sitename].plot(ax=ax[0], color = 'k')
    # # Observed
    lai_dict[sitename].plot(ax=ax[1], x=x_colname, y=y_colname, label=sitename, color = colors [idx])
    # Simulated
    headobs_coupled[obs_name.upper()].plot( ax=ax[2], label=f'{sitename}:{l,r,c}', color = colors [idx])


for idx, row in pj_network.iterrows():
    x, y = row.geometry.x, row.geometry.y
    label = row['Site']  # Change to another column if preferred
    ax[0].text(x, y, label, fontsize=9, ha='left', va='bottom', color='black')

# # Styling observed plot
# ax[1].legend(ncol = 2)
# ax[1].set_xlim('2024-11-1', '2024-11-10')
# ax[1].set_ylabel('Observed head (ft)')
# ax[1].set_title('Observed Heads \n 2024')
ax[1].set_ylim(-5, 25)


# # Styling simulated plot
# #ax[2].legend(ncol=1, loc = 'upper left')
# ax[2].set_xlim(start_date, end_date)
# ax[2].set_ylabel('Simulated head (ft)')
# ax[2].set_title(f'Simulated Heads \n 2018 \n{sim}')
ax[2].set_ylim(-5, 25)


# Common title
fig.suptitle('Simulated vs. Observed PJ', fontsize=16, y=1.05)

plt.show()


## Coupled v NOAA CHD

In [ ]:
lines_uncoupled = ':'
lines_coupled = '-'
lines_coupled_norech = '-.'
lines_uncoupled_timevary = '--'
site_colors = ['red', 'darkorange', 'gold','yellowgreen', 'deepskyblue','darkviolet', 'hotpink']


In [ ]:
start_date = pd.Timestamp('2018-09-19 00:00:00')
end_date = pd.Timestamp('2018-09-29 00:00:00')

site_data = {}
site_data_uncoupledNOAA = {}
fig, ax1 = plt.subplots(dpi=100, constrained_layout=True)

# SIMULATED HEADS
#-------------------------
for idx, (sitename,value) in enumerate(site_obsnames.items()):
    l,r,c = value['cellid']
    obs_name = value['obs_name'].upper()
    print(sitename, obs_name, l, r, c)

    # Simulated (coupled)
    headobs_coupled[obs_name].plot(
        x="Time",
        ax=ax1,
        linestyle=lines_coupled,
        label=f'{sitename}:{l,r,c}',
        color =site_colors[idx]
    )

    # Simulated (uncoupled)
    headobs_uncoupled[obs_name].plot(
        x="Time",
        ax=ax1,
        linestyle=lines_uncoupled,
        color =site_colors[idx]
    )

    # Simulated (uncoupled, time-varying CHD)
    headobs_uncoupled_NOAACHD[obs_name].plot(
        x="Time",
        ax=ax1,
        linestyle=lines_uncoupled_timevary,
        color =site_colors[idx]
    )

    site_data[sitename] = headobs_coupled[obs_name]
    site_data_uncoupledNOAA[sitename] = headobs_uncoupled_NOAACHD[obs_name]


# RECHARGE BARS
#----------------
lst_coupled['date'] = pd.to_datetime(lst_coupled['date'])

ax1a = ax1.twinx()
ax1a.bar(
    lst_coupled['date'],
    lst_coupled['RCHA_IN'],
    width=0.8,
    color='blue',
    alpha=0.9
)

ax1a.set_ylabel('Recharge')
ax1a.set_ylim(0, 1000000000)
ax1a.set_xlim(start_date, end_date)


# FORMAT SIMULATED PLOT
#----------------------
ax1.set_xlim(start_date, end_date)
ax1.set_ylabel('Simulated head (ft)')
ax1.set_ylim(-6, 25)

from matplotlib.lines import Line2D

style_handles = [
    Line2D([0], [0], color='k', lw=2, linestyle=lines_uncoupled, label='Uncoupled'),
        Line2D([0], [0], color='k', lw=2, linestyle=lines_uncoupled_timevary, label='Uncoupled (Time-varying CHD)'),
    Line2D([0], [0], color='k', lw=2, linestyle=lines_coupled, label='Coupled with DFLOW'),

]
ax1.legend(handles=style_handles, loc='upper right', fontsize=8)
plt.show()

# SAVE SIMULATED HEAD DATA
pj_sites_simulated = pd.DataFrame(site_data)
pj_sites_simulated_uncouplednoaa = pd.DataFrame(site_data_uncoupledNOAA)


In [ ]:
# Plat Mass Balance
#-------------------------
start_date = pd.Timestamp('2018-07-01 00:00:00')
end_date = pd.Timestamp('2018-09-30 00:00:00')
c_coupled = 'g'
c_noaa = 'b'
fig, axes = plt.subplots(2,1,figsize=(5,10))
axes[0].set_title('Total In')
lst_uncoupled_noaa.plot(x = 'date',y="TOTAL_IN", ax=axes[0], color = c_noaa)
lst_coupled.plot(x = 'date',y="TOTAL_IN", ax=axes[0], color = c_coupled)

axes[1].set_title('CHD In')
lst_ucoupled.plot(x = 'date',y="CHD_IN", ax=axes[1], color = c_noaa)

lst_coupled.plot(x = 'date',y="CHD_IN", ax=axes[1], color = c_coupled)



# lst.plot(y="TOTAL_OUT_NEG", ax=axes[0], color='r', label='Total Out (Negative)', alpha=0.7)
# lst.plot(y="PERCENT_DISCREPANCY", ax=axes[1])
# # Set labels, title, grid, and legend
for ax in axes:
    ax.set_xlim(start_date,end_date)
    ax.get_legend().remove()
#     ax.set_ylabel('')
#     ax.set_title('')
#     ax.grid()
#     ax.legend()
# plt.tight_layout()
# plt.show()

## No Surge, No Recharge

In [ ]:
# Calculate difference in head between two simulations
#---------------------------------------------------------
difference_recharge = pd.DataFrame()
difference_recharge['totim'] =  headobs_coupled['totim'].copy()

difference_surge = pd.DataFrame()
difference_surge['totim'] =  headobs_coupled['totim'].copy()

for idx, (sitename,value) in enumerate(site_obsnames.items()):
    obs_name = value['obs_name'].upper()
    difference_recharge[obs_name] = headobs_coupled[obs_name] - headobs_coupled_norech[obs_name]
    difference_surge[obs_name] = headobs_coupled[obs_name] - headobs_coupled_nosurge[obs_name]


In [ ]:
site_obsnames

In [ ]:
start_date = pd.Timestamp('2018-09-19 00:00:00')
end_date = pd.Timestamp('2018-10-19 00:00:00')
precip_startdate =pd.Timestamp('2018-09-25 00:00:00')
coupling_startdate =pd.Timestamp('2018-09-20 00:00:00')
rech_diff_dict = {}

# figure 1
fig = plt.figure(dpi = 150, constrained_layout=True)
ax = fig.add_subplot()
#figure 2
fig1 = plt.figure(dpi = 150,constrained_layout=True)
ax1 = fig1.add_subplot()
#figure 3
fig2 = plt.figure(dpi = 150,constrained_layout=True)
ax2 = fig2.add_subplot()

# SIMULATED HEADS
#-------------------------
for idx, (sitename,value) in enumerate(site_obsnames.items()):
    print('-----------------------')
    l,r,c = value['cellid']
    obs_name = value['obs_name'].upper()
    print(sitename)
    print('-----------------------')
    print(obs_name, l, r, c)

    # Simulated (coupled)
    headobs_coupled[obs_name].plot(
        x="Time",
        ax=ax,
        linestyle=lines_coupled,
        color=site_colors[idx]
    )
    # Simulated (coupled, no recharge )
    headobs_coupled_norech[obs_name].plot(
        x="Time",
        ax=ax,
        linestyle=lines_coupled_norech,
        color=site_colors[idx]
    )
    # Simulated (coupled, no surge )
    headobs_coupled_nosurge[obs_name].plot(
        x="Time",
        ax=ax,
        linestyle=lines_coupled,
        color='k',
        linewidth =1
    )

    if not sitename in ['MillCreek','Marina']:
        # Recharge driven head difference
        difference_recharge[obs_name].plot(x='Time',ax=ax1,color = site_colors[idx])
        diff_rech_data = difference_recharge[obs_name][precip_startdate:]
        print(f"Mean recharge difference after {precip_startdate} : {round(diff_rech_data.mean(), 2)}")
        print(f"Minimum recharge difference after {precip_startdate} : {round(diff_rech_data.min(), 2)}")
        print(f"Maximum recharge difference after {precip_startdate} : {round(diff_rech_data.max(), 2)}")
        # Storm surgedriven head difference
        
        # Storm surge head difference
        difference_surge[obs_name].plot(x='Time',ax=ax2,color = site_colors[idx])
        diff_surge_data = difference_surge[obs_name][coupling_startdate:]
        print(f'Mean surge difference after:  {round(diff_surge_data.mean(), 2)}')
        print(f'Minimum surge difference after: {round(diff_surge_data.min(), 2)}')
        print(f'Maximum surge difference after: {round(diff_surge_data.max(), 2)}')

        
    else:
        continue



# Plot recharge
for axs in [ax, ax1,ax2]:
    ax_twin = axs.twinx()
    ax_twin.bar(
        lst_coupled['date'],
        lst_coupled['RCHA_IN'],
        width=0.8,
        color='blue',
        alpha=0.9
    )
    ax_twin.set_ylabel('Recharge')
    ax_twin.set_ylim(0, 900000000)
    axs.set_xlim(start_date, end_date)
ax.set_ylabel('head (ft)')
ax1.set_ylabel('head from recharge (ft)')
ax2.set_ylabel('head from storm surge (ft)')

# # Styling
# ax[0].set_title('Simulated Heads\n2018')
# ax[1].set_title('Difference in heads attributed to recharge')
# ax[0].set_ylim(-6.5, 25)

from matplotlib.lines import Line2D

style_handles = [
    #Line2D([0], [0], color='k', lw=2, linestyle=lines_uncoupled, label='Uncoupled'),
    #Line2D([0], [0], color='k', lw=2, linestyle=lines_uncoupled_timevary, label='Uncoupled (Time-varying CHD)'),
    Line2D([0], [0], color='k', lw=2, linestyle=lines_coupled, label='Coupled with DFLOW'),
    Line2D([0], [0], color='k', lw=2, linestyle=lines_coupled_norech, label='Coupled with DFLOW, no recharge Modflow'),
    Line2D([0], [0], color='k', lw=2, linestyle=lines_coupled_norech, label='Coupled with DFLOW, no Storm surge')
]


ax.legend(handles=style_handles, loc='upper right', fontsize=8)
plt.show()


## Coastal Heads

In [ ]:
dflow_sim = r"D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\dflow-fm\highres\tides_atm_surge_2018/"

In [ ]:
x = []
y = []
x1 = []
y1 = []
sim ='run_15.00M'
datetimes  = "2018-09-29 00:00:00.000000" ,  "2018-09-29 06:00:00.000000"

r = 16
c = 39

fig,ax= plt.subplots(1,3, figsize = (10,6))
name_chd = obscells_CHD.loc[((obscells_CHD['lay'] == 0) & 
                                    (obscells_CHD['row']== r) & 
                                    (obscells_CHD['col'] == c)), 'chd_name'].iloc[0].upper().replace("'", "")
value_chd = obs_CHD[name_chd].loc[datetimes[0]]
#print(value_chd)
x1.append(0)
y1.append(value_chd)

for i in range(2,20):
    l = i
    name = obscells_heads.loc[
        (obscells_heads['lay'] == l) & 
        (obscells_heads['row'] == r) & 
        (obscells_heads['col'] == c),
        'obs_name'
    ].iloc[0].upper().replace("'", "")
    value = headobs_coupled[name].loc[datetimes[0]]
    #print(l,r,c,value)
    x.append(i)
    y.append(value)

    name_chd = obscells_CHD.loc[((obscells_CHD['lay'] == l) & 
                                      (obscells_CHD['row']== r) & 
                                      (obscells_CHD['col'] == c)), 'chd_name'].iloc[0].upper().replace("'", "")
    value_chd = obs_CHD[name_chd].loc[datetimes[0]]
    x1.append(i)
    y1.append(value_chd)



ax[0].plot(x, y, marker="o")  
ax[0].set_xlabel('Model Layer')
ax[0].set_ylabel('Head (ft)')
ax[0].set_title('head')
ax[1].plot(x1, y1, marker="o")
ax[1].set_xlabel('Model Layer')
#ax[1].set_yscale('log')
ax[1].set_ylabel('Coastal Flux')
ax[1].set_title('CHD flux')

mg_coastal.loc[((mg_coastal['row_pj'] == r) & (mg_coastal['col_pj'] == c))].plot(ax=ax[2])
ax[2].margins(10)
#cx.add_basemap(ax=ax[2], crs = mg_gdf.crs)

fig.suptitle(f'{r,c}\n{sim}')



In [ ]:
import re
#=================================================================
coords_UTM = []
start_date = pd.to_datetime('2018-09-20')
end_date = pd.to_datetime('2018-09-30')
colors = ['r','g','m','k']

sim = 'run_15.00M'
# CHD head Time Series 
fig, ax = plt.subplots(1, 3, figsize=(14, 7))

for i, (l, r, c) in enumerate([(0, 60, 60), (0, 64, 61), (0, 67, 59), (0, 67, 60)]):
    # assert the cells are CHDs
    assert ((obscells_CHD["lay"] == l) & (obscells_CHD["row"] == r) & (obscells_CHD["col"] == c)).any()

    # get the coordinated of the center of these cells (in UTM) to be used in the DFLOW-FM observation file to ensure that the heads are the same when coupling
    # ----------------------------------------------
    xs = [mg.xcellcenters[r,c]]
    xy = [mg.ycellcenters[r,c]]
    point = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xs, xy), crs="EPSG:4456")
    point.to_crs(epsg=32618, inplace=True)
    print(point, l,r,c)
    coords_UTM.append(point)


    # plot the map locations
    #-------------------------
    mg_gdf.loc[(mg_gdf['row_pj'] == r) & (mg_gdf['col_pj'] == c)].plot(
        ax=ax[0], color=colors[i]
    )

    # Plot the Modflow Head Observations
    #----------------------------------
    # get observation name
    obs_name = obscells_heads.loc[
        (obscells_heads['lay'] == 0) &
        (obscells_heads['row'] == r) &
        (obscells_heads['col'] == c), 
        'obs_name'
    ].iloc[0]
    obs_name = obs_name.upper()
    plot_df  =headobs_coupled[obs_name][start_date:end_date]
    # save out for liv
    #plot_df.to_csv(f'tides/tide_{l,r,c}.txt')
    # plot time series
    plot_df.plot(
        x="Time",
        label=f'{l, r, c}',
        ax=ax[1],
        color=colors[i]
    )

# Plot the Dflow water level observations
# ------------------------------------------
ds = xugrid.open_dataset(dflow_sim + r"run\output\FlowFM_his.nc")
# ds = ds.set_xindex("station_name")
station_dict = dict(zip(ds.stations.data, ds.stations['station_name'].data))
# loop through the observations that line up with the mf cell observations
for i,x in enumerate(range(23,27)): 
    station_name = station_dict[x]

    # If it's a byte string, decode to str
    if isinstance(station_name, (bytes, bytearray)):
        station_name = station_name.decode("utf-8", errors="ignore")

    # Strip trailing/leading whitespace
    station_name = station_name.strip()

    print(repr(station_name))  # should now look like '0,60,60'
    print(station_name)
    # convert data to feet
    data = ds.isel(stations=x)['waterlevel'] * 3.28 
    data.plot(ax=ax[2], label = station_name, color = colors[i])
    


# expand map extent by 10% (simple way)
ax[0].margins(0.8)
#cx.add_basemap(ax=ax[0], crs=mg_gdf.crs)

# format time series panel
start_date = pd.Timestamp('2018-09-24 00:00:00')
end_date = pd.Timestamp('2018-09-27 00:00:00')
ax[1].legend(ncol=2)
ax[1].set_title('Modflow water levels(ft)')
ax[1].set_ylim(-6, 6)
ax[1].set_xlim(start_date, end_date)

ax[2].set_title('DFlow-FM water levels (ft)')
ax[2].legend(ncol=2)
ax[2].set_ylim(-6, 6)
ax[2].set_xlim(start_date, end_date)

fig.suptitle(f'Water Levels \n {sim}')
# add basemap


In [ ]:
import re
#=================================================================
coords_UTM = []
start_date = pd.to_datetime('2018-09-20')
end_date = pd.to_datetime('2018-09-30')
sim = 'run_15.00M'
colors = ['r','g','m','k']
# CHD head Time Series 
fig, ax = plt.subplots(1, figsize=(14, 7))

for i, (l, r, c) in enumerate([(0, 60, 60), (0, 64, 61), (0, 67, 59), (0, 67, 60)]):
    # assert the cells are CHDs
    assert ((obscells_CHD["lay"] == l) & (obscells_CHD["row"] == r) & (obscells_CHD["col"] == c)).any()

    # get the coordinated of the center of these cells (in UTM) to be used in the DFLOW-FM observation file to ensure that the heads are the same when coupling
    # ----------------------------------------------
    xs = [mg.xcellcenters[r,c]]
    xy = [mg.ycellcenters[r,c]]
    point = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xs, xy), crs="EPSG:4456")
    point.to_crs(epsg=32618, inplace=True)
    print(point, l,r,c)
    coords_UTM.append(point)

    # Plot the Modflow Head Observations
    #----------------------------------
    # get observation name
    obs_name = obscells_heads.loc[
        (obscells_heads['lay'] == 0) &
        (obscells_heads['row'] == r) &
        (obscells_heads['col'] == c), 
        'obs_name'
    ].iloc[0]
    obs_name = obs_name.upper()
    plot_df  = headobs_coupled[obs_name][start_date:end_date]
    # save out for liv
    #plot_df.to_csv(f'tides/tide_{l,r,c}.txt')
    # plot time series
    plot_df.plot(
        x="Time",
        label=f'{l, r, c} - Modflow',
        ax=ax,
        color=colors[i]
    )
ax.legend()

# Plot the Dflow water level observations
# ------------------------------------------
ds = xugrid.open_dataset(dflow_sim + r"run\output\FlowFM_his.nc")
# ds = ds.set_xindex("station_name")
station_dict = dict(zip(ds.stations.data, ds.stations['station_name'].data))
# loop through the observations that line up with the mf cell observations
for i,x in enumerate(range(23,27)): 
    station_name = station_dict[x]

    # If it's a byte string, decode to str
    if isinstance(station_name, (bytes, bytearray)):
        station_name = station_name.decode("utf-8", errors="ignore")

    # Strip trailing/leading whitespace
    station_name = station_name.strip()

    print(repr(station_name))  # should now look like '0,60,60'
    print(station_name)
    # convert data to feet
    data = ds.isel(stations=x)['waterlevel'] * 3.28 
    data.plot(ax=ax, label = f'{station_name}- Dflow_FM', color = colors[i], linestyle = '--')
ax.legend()
fig.suptitle(f'Water Levels \n {sim}')
# add basemap


In [ ]:
ds = xugrid.open_dataset(r"D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\dflow-fm\midres\tides_atm_surge_2018\run\output\FlowFM_his.nc")
# ds = ds.set_xindex("station_name")
station_dict = dict(zip(ds.stations.data, ds.stations['station_name'].data))

fig,ax = plt.subplots(figsize = (8,6))
# loop through the observations that line up with the mf cell observations
for i,x in enumerate(range(23,27)): 
    station_name = station_dict[x]

    # If it's a byte string, decode to str
    if isinstance(station_name, (bytes, bytearray)):
        station_name = station_name.decode("utf-8", errors="ignore")

    # Strip trailing/leading whitespace
    station_name = station_name.strip()
    print(f'Dflow water level at Modflow Celll :{station_name}')
    # convert data to feet
    data = ds.isel(stations=x)['waterlevel'] * 3.28 

    data.plot(ax=ax, label = f'{station_name}- Dflow_FM')
start_date = pd.Timestamp('2018-09-24 00:00:00')
end_date = pd.Timestamp('2018-09-27 00:00:00')
ax.set_xlim(start_date, end_date)
ax.legend()
fig.suptitle(f'Water Levels')

## Head Contours


## Time Series

coastal heads : dflow versions

In [ ]:
sim_dict_headobs = {}

l,r,c = 0,60,62
obs_name = obscells_heads.loc[((obscells_heads['lay'] == l) & (obscells_heads['row'] == r) & (obscells_heads['col'] == c)), 'obs_name'].iloc[0]
obs_name =obs_name.upper()
obs_name

start_date = pd.Timestamp('2018-09-25 00:00:00')

fig,ax = plt.subplots()

headobs_coupled[obs_name].plot(x = "Time",ax=ax, label = f'{sim}')
ax.legend()
ax.set_xlim(start_date, headobs_coupled.index[-1])
plt.ylabel('head (ft)')
fig.suptitle(f'Head time series for cell {l,r,c}')
plt.show()

# geodataframe of cells to check
cell_location = pd.concat(
    [mg_gdf.loc[(mg_gdf['row_pj'] == r) & (mg_gdf['col_pj'] == c)]])
cell_location.explore(color ="MAGENTA")

## Depth to Water

In [ ]:
sim_pj = flopy.mf6.MFSimulation.load(sim_ws=coupled_ws, load_only=['oc'])
gwf = sim_pj.get_model("gwf")

In [ ]:

hdobj = gwf.output.head()
heads ={}
head_diff ={}
for i in hdobj.get_kstpkper():
    # Store daily heads
    heads[i] = hdobj.get_data(i)
    heads[i] = np.where(heads[i]==1.e+30 , np.nan, heads[i])

### 2017 Average DTW

In [ ]:
heads.keys()

In [ ]:
# 2017 average DTW
dtw = lay_arr_child[0] - heads[(0,1)]
# layer 1 DTW slice
data = dtw[0][50:100, 40:]
abs_max = np.nanmax(np.abs(data))
vmin = -abs_max
vmax = abs_max

# Larger, spaced figure
fig, ax = plt.subplots(1, 2, figsize=(10, 7), constrained_layout=True)

# --- First subplot: imshow diverging map ---
img = ax[0].imshow(data, vmin=vmin, vmax=vmax, cmap="RdBu_r")
c = plt.colorbar(img, ax=ax[0])
c.set_label("DTW (ft)")

# --- Second subplot: contour plot ---
import cartopy.crs as ccrs
original_crs = ccrs.epsg(4456)

c1 = ax[1].contour(
    mg.xcellcenters[50:100, 40:],
    mg.ycellcenters[50:100, 40:],
    data,
    linewidths=1.5,
    colors="k"
)
ax[1].clabel(c1, inline=True, fontsize=10, fmt='%1.1f')
# cx.add_basemap(ax[1], crs=original_crs)
fig.suptitle('2017 average DTW')
plt.show()


# Drain Observations 

In [ ]:
base_ws

In [ ]:
# GWF Obs 
#=========
obs_path  = fr'{coupled_ws}\outputs\drn_obs.csv'
df = flopy.utils.Mf6Obs(obs_path).get_dataframe(start_datetime="1-1-2017")
df_abs = df.abs()
drn_coupled = df_abs.merge(mf_tdis_df[['totim','Date']], how= 'left')

# uncoupled drain observations
obs_path = base_ws+r"\outputs\drn_obs.csv"
df = flopy.utils.Mf6Obs(obs_path).get_dataframe(start_datetime="1-1-2017")
df_abs = df.abs()
drn_uncoupled = df_abs.merge(mf_tdis_df[['totim','Date']], how= 'left')


# Info about Mill Creek drain observations
obscells_DRN =  pd.read_csv(fr'{coupled_ws}\gwf.drn.obs',skiprows=5, sep=r'  ', names = ['drn_name','head','cellid'])
obscells_DRN['cellid'] = obscells_DRN['cellid'].str.replace(r'\s+', ',', regex=True)
obscells_DRN[['lay', 'row', 'col']] = obscells_DRN['cellid'].str.split(',', expand=True)
obscells_DRN = obscells_DRN.iloc[:-1]
obscells_DRN[['lay', 'row', 'col']] = obscells_DRN[['lay', 'row', 'col']].astype(int)
obscells_DRN[['lay', 'row', 'col']]-=1

millcreek_cells =  mg_gdf.copy().merge(obscells_DRN, left_on=['row_pj','col_pj'],right_on = ['row', 'col'] )
millcreek_cells

In [ ]:
# Mill Creek Shapefile
millcreek = gpd.read_file(r"D:\LISS_GW\GW_Models\LISUS_conditionedmodels_BNB\2018\conditioned_model_2018_daily\_mfsetup\GIS\Creek_updated\Creek_BNB.shp")
millcreek.to_crs(epsg=4456, inplace=True)  # Update this line to assign the result to 'millcreek'


In [ ]:
date_ll = '2018-09-16'
date_ul = '2018-09-29'
totim_ll = mf_tdis_df.loc[mf_tdis_df['Date']== date_ll, 'totim'].iloc[0]
totim_ul = mf_tdis_df.loc[mf_tdis_df['Date']== date_ul, 'totim'].iloc[0]
print(totim_ll, totim_ul)


colors = ['red', 'darkorange', 'gold', 'green', 'cyan', 'blue', 'violet', 'magenta']
fig, axes = plt.subplots(1,2, figsize = (12,6)) 
for i in range(0, 8):
    drn_coupled.plot(x='totim', y=f'MILLCREEK{i}', ax=axes[0], color = colors[i], legend= False)
    drn_uncoupled.plot(x='totim', y=f'MILLCREEK{i}', ax=axes[0], linestyle = '--',color = colors[i], legend= False)

    millcreek_cells.loc[millcreek_cells['drn_name'] == f'millcreek{i}'].plot(color = colors[i], ax=axes[1])


axes[0].set_xlim(totim_ll,totim_ul)
axes[0].set_ylabel('ft³/day')
millcreek.plot(ax=axes[1], color = 'k')
#cx.add_basemap(crs = mg_gdf.crs, ax=axes[2])
plt.tight_layout()
plt.show()